# Mage-Flow-Turbo — PyTorch BF16 on Kaggle Tesla T4 ×2
## Public bilingual production demo / Demo production công khai song ngữ

### English
This notebook demonstrates one **single text-to-image trajectory** using one logical Mage-Flow-Turbo model distributed explicitly across **two Tesla T4 GPUs**. The runtime uses the qualified upstream PyTorch path with **BF16 dtype/materialization**, SDPA attention, one model load, explicit transformer placement, and fail-closed acceptance gates.

This public notebook is a reproducible demonstration and is not a replacement authority. The separate frozen R2G authority notebook/evidence remains the formal audit artifact.

### Tiếng Việt
Notebook này trình diễn **một luồng text-to-image duy nhất** bằng một model Mage-Flow-Turbo logic được đặt tường minh trên **hai GPU Tesla T4**. Runtime sử dụng pipeline PyTorch upstream đã được kiểm định với **BF16 dtype/materialization**, SDPA attention, chỉ load model một lần, placement transformer tường minh và các acceptance gate fail-closed.

Notebook public này dùng để demo và tái lập kết quả. Notebook/evidence R2G đã đóng băng vẫn là artifact kiểm định chính thức.

| Setting / Thiết lập | Value / Giá trị |
|---|---|
| Runtime | Upstream PyTorch |
| Precision / Độ chính xác | BF16 dtype/materialization |
| GPU | Tesla T4 ×2 |
| Topology | One logical model / one T2I trajectory |
| Transformer split | block 0 → GPU0; blocks 1–11 → GPU1 |
| Attention | SDPA |
| Output | 512×512 RGB |


## Before you run / Trước khi chạy

### English
Prepare a **fresh Kaggle Notebook** with:

1. **Accelerator:** Tesla T4 ×2.
2. **Internet:** ON. The notebook clones the public source repository and fetches the exact pinned upstream Mage source plus one pinned bootstrap wheel.
3. **Kaggle Model:** attach `dangkhoa2016/mage-flow-community-mage-flow-turbo`.
   Kaggle should mount it read-only at:
   `/kaggle/input/models/dangkhoa2016/mage-flow-community-mage-flow-turbo/pytorch/default/1`
4. **Dataset:** no additional dataset is required for this text-to-image demo.
5. Start from a fresh kernel and use **Run All exactly once**.

You do **not** need to upload any project ZIP, identity JSON, or source archive. The project source is cloned automatically from GitHub into `/kaggle/working`.

### Tiếng Việt
Hãy chuẩn bị một **Kaggle Notebook mới** với:

1. **Accelerator:** Tesla T4 ×2.
2. **Internet:** ON. Notebook sẽ tự clone source public từ GitHub và lấy đúng upstream Mage source cùng một bootstrap wheel đã pin.
3. **Kaggle Model:** attach `dangkhoa2016/mage-flow-community-mage-flow-turbo`.
   Kaggle dự kiến mount read-only tại:
   `/kaggle/input/models/dangkhoa2016/mage-flow-community-mage-flow-turbo/pytorch/default/1`
4. **Dataset:** demo text-to-image này không cần dataset bổ sung.
5. Dùng fresh kernel và **Run All đúng một lần**.

Bạn **không cần** upload project ZIP, identity JSON hay source archive nào. Source của dự án được notebook tự động clone từ GitHub vào `/kaggle/working`.


## Architecture / Kiến trúc

### English
The text encoder runs on GPU0. Transformer block 0 stays on GPU0; blocks 1–11 and the output head run on GPU1. The transformer result returns to GPU0 for the scheduler/latent path, then the VAE input is transferred to GPU1 for decoding.

### Tiếng Việt
Text encoder chạy trên GPU0. Transformer block 0 nằm trên GPU0; các block 1–11 và output head nằm trên GPU1. Output của transformer quay về GPU0 cho scheduler/latent path, sau đó VAE input được chuyển sang GPU1 để decode.

```text
Prompt / Prompt
   |
Text Encoder / Bộ mã hóa văn bản
   cuda:0
   |
Transformer
   |
block 0 ----------------------------- cuda:0
   |
activation 0 -> 1 ------------------>
   |
blocks 1..11 ------------------------ cuda:1
norm_out / proj_out ----------------- cuda:1
   |
transformer output 1 -> 0 ---------->
   |
Latent / scheduler path ------------- cuda:0
   |
VAE input 0 -> 1 ------------------->
   |
VAE --------------------------------- cuda:1
   |
512×512 RGB image
```

Across four denoising steps, the block sequence `[0..11]` repeats four times. That repetition is expected; it is not a duplicate-block error.

Qua bốn denoising step, chuỗi block `[0..11]` lặp lại bốn lần. Đây là hành vi đúng, không phải lỗi duplicate block.


## Step 1 — Clone the pinned public source / Clone source public đã pin

### English
This cell clones the public repository at release tag `v1.0.0` into a fresh `/kaggle/working` directory, verifies the origin URL and exact release tag, records the resolved Git commit SHA, then establishes `PROJECT_ROOT` and `sys.path`.

**Expected:** `PROJECT_SOURCE_BOOTSTRAP=PASS`.  
**If it fails:** stop; do not delete a stale checkout and retry inside the same publication run.

### Tiếng Việt
Cell này clone public repository tại release tag `v1.0.0` vào một thư mục mới trong `/kaggle/working`, xác minh origin URL và đúng release tag, ghi lại Git commit SHA thực tế, sau đó mới thiết lập `PROJECT_ROOT` và `sys.path`.

**Kỳ vọng:** `PROJECT_SOURCE_BOOTSTRAP=PASS`.  
**Nếu lỗi:** dừng lại; không xóa checkout cũ rồi chạy lại trong cùng publication run.


In [ ]:
# --- Public Git source bootstrap ---
# No publication ZIP or identity JSON is required.

import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/dangkhoa2016/Mage-Flow-Turbo-PyTorch-BF16-on-T4x2-GPU.git"
RELEASE_REF = "v1.0.0"
PROJECT_ROOT = Path("/kaggle/working/Mage-Flow-Turbo-PyTorch-BF16-on-T4x2-GPU").resolve()

if PROJECT_ROOT.exists():
    raise RuntimeError(
        "PROJECT_SOURCE_TARGET_STALE: expected a fresh checkout target; "
        f"already exists: {PROJECT_ROOT}"
    )

subprocess.run(
    [
        "git", "clone",
        "--branch", RELEASE_REF,
        "--single-branch",
        REPOSITORY_URL,
        str(PROJECT_ROOT),
    ],
    check=True,
)

origin = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "remote", "get-url", "origin"],
    text=True,
).strip()
resolved_tag = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "describe", "--tags", "--exact-match"],
    text=True,
).strip()
resolved_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

if origin != REPOSITORY_URL:
    raise RuntimeError(f"PROJECT_SOURCE_ORIGIN_MISMATCH: {origin}")
if resolved_tag != RELEASE_REF:
    raise RuntimeError(
        f"PROJECT_SOURCE_RELEASE_REF_MISMATCH: expected={RELEASE_REF} actual={resolved_tag}"
    )
if not (PROJECT_ROOT / "mage_t4x2").is_dir():
    raise RuntimeError("PROJECT_SOURCE_LAYOUT_INVALID: mage_t4x2 missing")
if not (PROJECT_ROOT / "authority" / "r2g-runtime-baseline.json").is_file():
    raise RuntimeError("PROJECT_SOURCE_LAYOUT_INVALID: R2G baseline missing")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("REPOSITORY_URL=", origin)
print("RELEASE_REF=", resolved_tag)
print("RESOLVED_COMMIT=", resolved_commit)
print("PROJECT_ROOT=", PROJECT_ROOT)
print("PROJECT_SOURCE_BOOTSTRAP=PASS")


## Step 2 — Environment and GPU inventory / Môi trường và kiểm kê GPU

### English
This cell checks Python/PyTorch/CUDA and requires **exactly two Tesla T4 GPUs**. It does not load the model.

**Expected:** `GPU count = 2`, and both devices contain `T4`.

### Tiếng Việt
Cell này kiểm tra Python/PyTorch/CUDA và yêu cầu **đúng hai GPU Tesla T4**. Cell chưa load model.

**Kỳ vọng:** `GPU count = 2` và cả hai GPU đều là `T4`.


In [ ]:
# --- Environment and GPU inventory ---
# Do not load or preload the model in this cell.
# PROJECT_ROOT / sys.path were already established by the source-bootstrap cell.

import platform
import sys

assert PROJECT_ROOT is not None and (PROJECT_ROOT / "mage_t4x2").is_dir(), (
    "PROJECT_ROOT must be established by the source-bootstrap cell"
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from mage_t4x2.environment import cuda_inventory

inventory = cuda_inventory()
print(f"Python      : {platform.python_version()}")
print(f"PyTorch     : {torch.__version__} (CUDA {torch.version.cuda}) available={torch.cuda.is_available()}")
print(f"GPU count   : {len(inventory)}")
for dev in inventory:
    print(f"  GPU{dev['index']}  {dev['name']}  compute {dev['compute_capability']}  vram {dev['total_vram_bytes'] / 2**30:.1f} GiB")
assert len(inventory) == 2 and all("T4" in dev["name"] for dev in inventory), "exactly two Tesla T4 GPUs required; fail closed"

## Step 3 — R2G runtime integrity / Tính toàn vẹn runtime R2G

### English
Before public inference, the notebook verifies the frozen R2G runtime baseline. This protects the already-qualified runtime from silent modification.

**Expected:** `R2G_RUNTIME_BASELINE_INTEGRITY=PASS`, `ROWS=31`, `MISMATCHES=0`.

### Tiếng Việt
Trước khi inference public, notebook xác minh R2G runtime baseline đã đóng băng. Bước này bảo vệ runtime đã được kiểm định khỏi thay đổi ngoài ý muốn.

**Kỳ vọng:** `R2G_RUNTIME_BASELINE_INTEGRITY=PASS`, `ROWS=31`, `MISMATCHES=0`.


In [ ]:
# --- Source / runtime integrity ---
# Read-only verification of the R2G runtime baseline (authority/r2g-runtime-baseline.json).

from mage_t4x2.r2g_runtime_baseline import verify_r2g_runtime_baseline

baseline = verify_r2g_runtime_baseline(PROJECT_ROOT)
print(f"R2G_RUNTIME_BASELINE_INTEGRITY={baseline['status']}")
print(f"ROWS={len(baseline['entries'])}")
print(f"MISMATCHES={sum(1 for e in baseline['entries'] if e['status'] != 'MATCH')}")
print(f"manifest_sha256={baseline.get('manifest_sha256')}")
assert baseline["status"] == "PASS", "runtime baseline integrity failed; stop and preserve evidence"


## Step 4 — Reproducibility configuration and model resolution / Cấu hình tái lập và xác định model

### English
This cell defines the canonical demo settings and resolves the owner-qualified Kaggle model path. The model must come from the read-only Kaggle input mount; no remote fallback is accepted.

| Setting | Value |
|---|---|
| Model | `dangkhoa2016/mage-flow-community-mage-flow-turbo` |
| Prompt | `a red fox in a snowy forest at golden hour, high detail` |
| Resolution | 512×512 |
| Steps | 4 |
| CFG | 1.0 |
| Seed | 42 |
| split_block / num_blocks | 1 / 12 |

**Expected:** `MODEL_SOURCE=LOCAL_ATTACHMENT` and `MODEL_REQUIRED_FILES=PASS`.

### Tiếng Việt
Cell này khai báo cấu hình canonical của demo và xác định đường dẫn Kaggle Model có đầy đủ owner namespace. Model phải đến từ Kaggle input read-only; không chấp nhận remote fallback.

**Kỳ vọng:** `MODEL_SOURCE=LOCAL_ATTACHMENT` và `MODEL_REQUIRED_FILES=PASS`.


In [ ]:
# --- Demo configuration object ---
# Human-readable config; the model is NOT loaded in this cell.

import dataclasses
import os
from pathlib import Path

from mage_t4x2 import constants as C
from mage_t4x2.model_provenance import ModelPathResolver
from public_demo.runner import CANONICAL_PROMPT


@dataclasses.dataclass(frozen=True)
class PublicDemoConfig:
    prompt: str = CANONICAL_PROMPT
    width: int = C.RESOLUTION_WIDTH
    height: int = C.RESOLUTION_HEIGHT
    steps: int = C.STEPS
    cfg_scale: float = C.CFG
    seed: int = C.SEED
    num_blocks: int = 12
    split_block: int = 1


CONFIG = PublicDemoConfig()
print("PROMPT", CONFIG.prompt)
print("SEED", CONFIG.seed, "| STEPS", CONFIG.steps, "| CFG", CONFIG.cfg_scale)
print("RESOLUTION", f"{CONFIG.width}x{CONFIG.height}")

# Resolve the pinned local Kaggle model attachment (read-only /kaggle/input; path resolution only).
#
# Canonical path is built from the FULL owner-qualified model ID:
#   /kaggle/input/models/dangkhoa2016/mage-flow-community-mage-flow-turbo/pytorch/default/1
# The owner namespace is never dropped. KAGGLE_MODELS_INPUT_ROOT exists solely so
# CPU tests can simulate a fresh Kaggle layout without touching /kaggle.
MODELS_INPUT_ROOT = Path(
    os.environ.get("KAGGLE_MODELS_INPUT_ROOT", "/kaggle/input/models")
).resolve()
canonical_model_path = (
    MODELS_INPUT_ROOT / C.MODEL_ID / "pytorch" / "default" / "1"
).resolve()

# Canonical owner-qualified model path only; no basename-only or remote fallback.
resolution = ModelPathResolver(
    candidates=[str(canonical_model_path)],
    required_rel_files=[
        "model_index.json",
        "transformer/config.json",
        "transformer/diffusion_pytorch_model.safetensors",
        "scheduler/scheduler_config.json",
    ],
).resolve(fallback_id=None)
MODEL_PATH = resolution["selected_path"]
MODEL_SOURCE = resolution["model_source"]
model_required_files_ok = bool(resolution["required_files_present"])

# Fail-closed model contract: LOCAL_ATTACHMENT on the owner-qualified canonical
# path with every required file present; no remote fallback is ever accepted.
assert MODEL_SOURCE == "LOCAL_ATTACHMENT", "expected the pinned local Kaggle model attachment"
assert Path(MODEL_PATH).resolve() == canonical_model_path, "MODEL_PATH_CANONICAL=FAIL: owner-qualified path required"
assert model_required_files_ok, "MODEL_REQUIRED_FILES=FAIL"

print("MODEL_ID", C.MODEL_ID)
print("MODEL_PATH", MODEL_PATH)
print("MODEL_SOURCE", MODEL_SOURCE)
print("MODEL_REQUIRED_FILES", "PASS" if model_required_files_ok else "FAIL")


## Live acceptance gates / Các acceptance gate khi chạy thật

### English
The runner verifies live evidence rather than trusting configuration intent. Required gates include two T4 GPUs, one model load, one T2I trajectory, block routing, cross-GPU transfers, SDPA after model load and immediately before inference, BF16 materialization, no CPU fallback, and a valid 512×512 RGB image.

### Tiếng Việt
Runner xác minh evidence thực tế thay vì chỉ tin vào cấu hình khai báo. Các gate bắt buộc gồm hai T4, chỉ một lần load model, một T2I trajectory, block routing, transfer giữa hai GPU, SDPA sau model load và ngay trước inference, BF16 materialization, không CPU fallback và ảnh RGB 512×512 hợp lệ.


## Step 5 — Run the public demo / Chạy public demo

### English
This is the only heavy cell. It loads the model exactly once and runs exactly one T2I trajectory. Failure diagnostics (`first_failed_gate`, `error`, summary/output paths) are printed before the assertion so a failed run remains debuggable.

**Do not rerun this cell alone** in a publication proof. If it fails, preserve the failed attempt.

### Tiếng Việt
Đây là cell nặng duy nhất. Cell load model đúng một lần và chạy đúng một T2I trajectory. Các diagnostic (`first_failed_gate`, `error`, đường dẫn summary/output) được in trước assertion để failed run vẫn có đủ evidence.

**Không chạy lại riêng cell này** trong publication proof. Nếu lỗi, hãy giữ nguyên failed attempt.


In [ ]:
# --- Execution: run the public demo ---
# This single cell loads the model exactly once and runs exactly one T2I trajectory.

from public_demo.runner import run_public_demo

DEMO_RESULT = run_public_demo(
    project_root=str(PROJECT_ROOT),
    model_path=MODEL_PATH,
    prompt=CONFIG.prompt,
    seed=CONFIG.seed,
    steps=CONFIG.steps,
    width=CONFIG.width,
    height=CONFIG.height,
    cfg_scale=CONFIG.cfg_scale,
    num_blocks=CONFIG.num_blocks,
    split_block=CONFIG.split_block,
)

print("run_id :", DEMO_RESULT.get("run_id"))
print("verdict:", DEMO_RESULT.get("status"))
print("first_failed_gate:", DEMO_RESULT.get("first_failed_gate"))
print("error:", DEMO_RESULT.get("error"))
print("summary:", DEMO_RESULT.get("summary_path"))
print("output :", DEMO_RESULT.get("output_dir"))

assert DEMO_RESULT["status"] == "PASS", "public demo failed; stop and preserve evidence"


## Step 6 — Concise live result summary / Tóm tắt kết quả live

### English
This cell reads the `summary.json` produced by the runner and prints the key hardware, routing, precision, safety, and output facts.

### Tiếng Việt
Cell này đọc `summary.json` do runner tạo ra và in các thông tin chính về hardware, routing, precision, safety và output.


In [ ]:
# --- Concise result summary ---
# Read the public-demo summary.json produced by the runner (tabular, no raw dumps).

import json

with open(DEMO_RESULT["summary_path"], encoding="utf-8") as fh:
    SUMMARY = json.load(fh)

assert SUMMARY["status"] in ("PASS", "FAIL"), "summary.status must be a live runner verdict"

h, r = SUMMARY["hardware"], SUMMARY["runtime"]
rg, p, s, o = SUMMARY["routing"], SUMMARY["precision"], SUMMARY["safety"], SUMMARY["output"]

rows = [
    ("GPU count", h["device_count"]),
    ("GPU0", h["gpu0"]),
    ("GPU1", h["gpu1"]),
    ("model_load_count", r["model_load_count"]),
    ("split_block", r["split_block"]),
    ("num_blocks", r["num_blocks"]),
    ("GPU0 participation", "PASS" if rg["gpu0_participation"] else "FAIL"),
    ("GPU1 participation", "PASS" if rg["gpu1_participation"] else "FAIL"),
    ("transformer invocation count", f"{rg['observed_transformer_invocations']} / {rg['expected_transformer_invocations']}"),
    ("0->1 transfer count", rg["transfer_0_to_1_count"]),
    ("1->0 return count", rg["transformer_return_1_to_0_count"]),
    ("VAE 0->1 count", rg["vae_input_transfer_0_to_1_count"]),
    ("CPU fallback observed", s["cpu_fallback_observed"]),
    ("output exists", o["exists"]),
    ("output width", o["width"]),
    ("output height", o["height"]),
    ("output mode", o["mode"]),
    ("BF16 status", p["status"]),
]
width = max(len(k) for k, _ in rows)
for k, v in rows:
    print(f"{k.ljust(width)}  {v}")


## Step 7 — Device placement / Phân bố thiết bị

### English
This table shows the **applied** component/block placement from the live run, not merely the intended configuration.

### Tiếng Việt
Bảng này hiển thị placement **đã thực sự được áp dụng** trong live run, không chỉ là cấu hình dự kiến.


In [ ]:
# --- Device placement table ---
# Read-only rendering of the explicit dual-GPU placement recorded by the run.

placement = SUMMARY["placement"]
print(f"{'Component / block':<24}{'Device'}")
print("-" * 36)
for key in placement:
    print(f"{key:<24}{placement[key]}")


## Step 8 — Multistep routing evidence / Evidence routing qua nhiều step

### English
Telemetry is reduced per transformer invocation. Each of the four invocations must execute blocks `0..11` exactly once in order, with the expected GPU0→GPU1 boundary and GPU1→GPU0 return.

### Tiếng Việt
Telemetry được tổng hợp theo từng transformer invocation. Mỗi một trong bốn invocation phải chạy các block `0..11` đúng một lần theo đúng thứ tự, với boundary GPU0→GPU1 và đường return GPU1→GPU0 như dự kiến.


In [ ]:
# --- Multistep routing summary ---
# Per-invocation block integrity derived from the captured telemetry (evidence, not config intent).

from pathlib import Path
import json
from mage_t4x2.evidence_reducers import adjudicate_multistep_block_integrity

telemetry_path = Path(DEMO_RESULT["output_dir"]) / "telemetry.jsonl"
records = [json.loads(line) for line in telemetry_path.open(encoding="utf-8") if line.strip()]
adjudicated = adjudicate_multistep_block_integrity(
    records,
    run_id=SUMMARY["run_id"],
    num_blocks=SUMMARY["runtime"]["num_blocks"],
    expected_invocations=SUMMARY["routing"]["expected_transformer_invocations"],
)

print(f"{'Invocation':<12}{'Block sequence':<16}{'0->1':<6}{'1->0':<6}Status")
for i, seq in enumerate(adjudicated["invocation_block_sequences"], start=1):
    canonical = seq == list(range(SUMMARY["runtime"]["num_blocks"]))
    text = "0.." + str(seq[-1]) if seq else "-"
    print(f"{i:<12}{text:<16}{'yes':<6}{'yes':<6}{'PASS' if canonical else 'FAIL'}")
print()
print("NO_SKIPPED_BLOCKS=" + ("PASS" if adjudicated["NO_SKIPPED_BLOCKS"] else "FAIL"))
print("NO_DUPLICATED_BLOCKS_WITHIN_INVOCATION=" + ("PASS" if adjudicated["NO_DUPLICATED_BLOCKS"] else "FAIL"))
print("BLOCK_ORDER_VALID=" + ("PASS" if adjudicated["BLOCK_ORDER_VALID"] else "FAIL"))


## Step 9 — BF16 materialization / Xác minh BF16 materialization

### English
This cell audits the live component dtypes for `text_encoder`, `transformer`, and `vae`.

The supported claim is **BF16 dtype/materialization + functional upstream PyTorch execution on Tesla T4**. This is **not** a claim of native BF16 Tensor Core acceleration on T4.

### Tiếng Việt
Cell này kiểm tra dtype thực tế của `text_encoder`, `transformer` và `vae`.

Phát biểu được hỗ trợ là **BF16 dtype/materialization + functional upstream PyTorch execution on Tesla T4**. Đây **không phải** tuyên bố rằng T4 có native BF16 Tensor Core acceleration.


In [ ]:
# --- BF16 materialization ---
# Component-level dtype audit after the run (authority schema: text_encoder / transformer / vae).

p = SUMMARY["precision"]
for component in ("text_encoder", "transformer", "vae"):
    label = p.get(component, "unknown")
    status = "PASS" if str(label).endswith("bfloat16") else "FAIL"
    print(f"{component:<14}{label:<24}{status}")
print()
print("unexpected_floating_dtypes=", p.get("unexpected_floating_dtypes", []))
print()
print("BF16 dtype/materialization + functional upstream PyTorch execution on Tesla T4")


## Step 10 — Generated image / Ảnh được sinh ra

### English
This cell displays the real 512×512 RGB image produced by the current run and prints the reproducibility settings plus output SHA256.

### Tiếng Việt
Cell này hiển thị ảnh RGB 512×512 thực tế được tạo bởi lần chạy hiện tại và in cấu hình tái lập cùng SHA256 của output.


In [ ]:
# --- Display generated image ---
# The actual 512x512 RGB image produced by this public demo run.

from IPython.display import Image as IPImage, display

display(IPImage(filename=SUMMARY["output"]["path"]))
print("seed      :", CONFIG.seed)
print("steps     :", CONFIG.steps)
print("CFG       :", CONFIG.cfg_scale)
print("resolution:", f"{CONFIG.width}x{CONFIG.height}")
print("SHA256    :", SUMMARY["output"]["sha256"])


## What the verified R2G authority run established / Những gì R2G authority đã xác minh

### English
The formal R2G dual-T4 authority run is already closed and preserved separately. Historical facts:

- `SESSION_ID=321a1a739d7d`
- `G1=PASS`
- `G2=PASS`
- `MODEL_LOAD_COUNT=1`
- `G3_G6_EXECUTED=NO`

This public notebook does not replace or rewrite that authority evidence.

### Tiếng Việt
R2G authority chính thức cho dual-T4 đã được đóng và lưu riêng. Các fact lịch sử:

- `SESSION_ID=321a1a739d7d`
- `G1=PASS`
- `G2=PASS`
- `MODEL_LOAD_COUNT=1`
- `G3_G6_EXECUTED=NO`

Notebook public này không thay thế hoặc ghi đè authority evidence đó.


## Reproducibility and limitations / Tái lập và giới hạn

### English
- Public source repository: `dangkhoa2016/Mage-Flow-Turbo-PyTorch-BF16-on-T4x2-GPU`
- Public notebook source ref: release tag `v1.0.0`
- Model: `dangkhoa2016/mage-flow-community-mage-flow-turbo`
- Model revision: `65bb3500f0da9df6a41ec6383716fc02cf014773`
- Upstream Mage commit: `76bec2bb3818863f470de7e867c2dc7f1d0bfd83`
- Upstream `mage_flow` tree: `946b91bcb2cac75e6cfe8399f0f7f330a2280adf`
- Runtime baseline: `authority/r2g-runtime-baseline.json`
- Hardware: exactly Tesla T4 ×2
- Attention: SDPA, reasserted after model construction and immediately before inference
- Bootstrap dependency: `loguru==0.7.3`, wheel SHA-256 pinned by the repository
- CPU fallback: forbidden
- No `flash_attn` dependency is required for this T4 path
- Internet must be ON for public Git/PyPI bootstrap
- The supported precision wording is **BF16 dtype/materialization + functional upstream PyTorch execution on Tesla T4**

### Tiếng Việt
- Public source repository: `dangkhoa2016/Mage-Flow-Turbo-PyTorch-BF16-on-T4x2-GPU`
- Source ref của public notebook: release tag `v1.0.0`
- Model: `dangkhoa2016/mage-flow-community-mage-flow-turbo`
- Model revision: `65bb3500f0da9df6a41ec6383716fc02cf014773`
- Upstream Mage commit: `76bec2bb3818863f470de7e867c2dc7f1d0bfd83`
- Upstream `mage_flow` tree: `946b91bcb2cac75e6cfe8399f0f7f330a2280adf`
- Runtime baseline: `authority/r2g-runtime-baseline.json`
- Phần cứng: đúng Tesla T4 ×2
- Attention: SDPA, được reassert sau model construction và ngay trước inference
- Bootstrap dependency: `loguru==0.7.3`, SHA-256 wheel được pin trong repository
- CPU fallback: không cho phép
- T4 path này không yêu cầu dependency `flash_attn`
- Public Git/PyPI bootstrap yêu cầu Internet ON
- Cách mô tả precision được hỗ trợ là **BF16 dtype/materialization + functional upstream PyTorch execution on Tesla T4**


## Step 11 — Final live verdict / Kết luận live cuối cùng

### English
The final verdict below is derived only from the current live `SUMMARY`. A publication PASS requires every mandatory gate to pass and `SUMMARY["status"] == "PASS"`.

### Tiếng Việt
Kết luận cuối bên dưới chỉ được suy ra từ `SUMMARY` của lần chạy hiện tại. Publication chỉ PASS khi toàn bộ gate bắt buộc đều PASS và `SUMMARY["status"] == "PASS"`.


In [ ]:
# --- Final summary: public-demo verdict derived from live SUMMARY ---
# No static public-demo verdict values exist in this notebook: every line below is
# derived at runtime from the live SUMMARY (summary.json) produced by the runner
# for the current run, and fails closed if required fields are absent.

required = {
    "two_t4": (
        SUMMARY["hardware"]["device_count"] == 2
        and "T4" in SUMMARY["hardware"]["gpu0"]
        and "T4" in SUMMARY["hardware"]["gpu1"]
    ),
    "single_model_load": SUMMARY["runtime"]["model_load_count"] == 1,
    "single_trajectory_dual_gpu": (
        SUMMARY["routing"]["gpu0_participation"] is True
        and SUMMARY["routing"]["gpu1_participation"] is True
        and SUMMARY["routing"]["single_t2i_instance"] is True
    ),
    "block_routing": (
        SUMMARY["routing"]["block_order_valid"] is True
        and SUMMARY["routing"]["no_skipped_blocks"] is True
        and SUMMARY["routing"]["no_duplicated_blocks_within_invocation"] is True
    ),
    "cross_gpu_transfers": (
        SUMMARY["routing"]["transfer_0_to_1_count"] == 4
        and SUMMARY["routing"]["transformer_return_1_to_0_count"] == 4
        and SUMMARY["routing"]["vae_input_transfer_0_to_1_count"] == 1
    ),
    "bf16": SUMMARY["precision"]["status"] == "PASS",
    "no_cpu_fallback": SUMMARY["safety"]["cpu_fallback_observed"] is False,
    "output": (
        SUMMARY["output"]["exists"] is True
        and SUMMARY["output"]["width"] == 512
        and SUMMARY["output"]["height"] == 512
        and SUMMARY["output"]["mode"] == "RGB"
        and SUMMARY["output"]["nan"] is False
        and SUMMARY["output"]["inf"] is False
    ),
}

overall = all(required.values()) and SUMMARY["status"] == "PASS"


def verdict(ok):
    return "PASS" if ok else "FAIL"


print("Public demo result")
print("------------------")
print("Two Tesla T4 GPUs:", verdict(required["two_t4"]))
print("Single logical model load:", verdict(required["single_model_load"]))
print("Single T2I trajectory across both GPUs:", verdict(required["single_trajectory_dual_gpu"]))
print("Explicit block routing:", verdict(required["block_routing"]))
print("Cross-GPU transfers:", verdict(required["cross_gpu_transfers"]))
print("BF16 materialization:", verdict(required["bf16"]))
print("CPU fallback:", "NOT OBSERVED" if required["no_cpu_fallback"] else "OBSERVED")
print("512×512 RGB output:", verdict(required["output"]))
print("PUBLIC_DEMO_FINAL_VERDICT=" + verdict(overall))

assert overall, "PUBLIC_DEMO_FINAL_VERDICT=FAIL"
